In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point, box
import h3

### Wind data frame as h3 hexagons

In [ ]:
H3_RESOLUTION = 8
WIND_GEOJSON_PATH = "/Users/lukepitstick/Projects/hackathon/renewably/src/python/ml/data/geoJSON/wind_us_farms.geojson"
SOLAR_GEOJSON_PATH = "/Users/lukepitstick/Projects/hackathon/renewably/src/python/ml/data/geoJSON/solar_us_farms.geojson"

In [3]:
def load_dataset(path: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    # ensure WGS84
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf
 
 
def turbine_points_to_h3(gdf: gpd.GeoDataFrame, resolution: int) -> set[str]:
    """Convert each turbine point to its containing H3 cell index."""
    h3_indices = set()
    for _, row in gdf.iterrows():
        pt = row.geometry
        idx = h3.latlng_to_cell(pt.y, pt.x, resolution)
        h3_indices.add(idx)
    return h3_indices

def generate_conus_h3_cells(resolution: int) -> set[str]:
    """
    Generate all H3 cells covering the contiguous US.
 
    Strategy: use h3.polygon_to_cells() with a rough CONUS bounding polygon.
    This is the fast way — no need to iterate a manual grid.
    """
    # Rough CONUS boundary (simplified polygon)
    # For production, use a proper US boundary shapefile
    conus_coords = [
        (-125.0, 24.5), (-66.0, 24.5), (-66.0, 49.5),
        (-125.0, 49.5), (-125.0, 24.5)
    ]
    # h3.polygon_to_cells expects an H3 Polygon:
    # outer ring as list of (lat, lng) tuples — NOTE: h3 uses (lat, lng) not (lng, lat)
    outer_ring = [(lat, lng) for lng, lat in conus_coords]
 
    cells = h3.polygon_to_cells(
        h3.LatLngPoly(outer_ring),
        res=resolution
    )
    return set(cells)

def build_grid_dataframe(
    all_cells: set[str],
    turbine_cells: set[str]
) -> pd.DataFrame:
    """
    Build a DataFrame where each row is an H3 cell with:
      - h3_index: the cell ID
      - lat, lng: cell center
      - has_turbine: 1/0
    """
    records = []
    for cell in all_cells:
        lat, lng = h3.cell_to_latlng(cell)
        records.append({
            "h3_index": cell,
            "lat": lat,
            "lng": lng,
            "has_turbine": int(cell in turbine_cells),
        })
    return pd.DataFrame(records)

In [15]:
wind_geojson = load_dataset(WIND_GEOJSON_PATH)

In [16]:
wind_h3_cells = turbine_points_to_h3(wind_geojson, H3_RESOLUTION)
wind_df = build_grid_dataframe(generate_conus_h3_cells(H3_RESOLUTION), wind_h3_cells)


In [6]:
wind_df['has_turbine'].value_counts()

has_turbine
0    2749977
1      15169
Name: count, dtype: int64

### Solar data frame as h3 hexagons

In [20]:
def solar_polygons_to_h3(gdf: gpd.GeoDataFrame, resolution: int) -> set[str]:
    """Find all H3 cells that overlap any solar farm polygon."""
    h3_indices = set()
    for _, row in gdf.iterrows():
        geom = row.geometry

        # h3.polygon_to_cells() fills a polygon with H3 cells
        # Need to convert shapely coords to h3's (lat, lng) format
        exterior = [(y, x) for x, y in geom.exterior.coords]
        holes = [
            [(y, x) for x, y in hole.coords]
            for hole in geom.interiors
        ]

        poly = h3.LatLngPoly(exterior, *holes)
        cells = h3.polygon_to_cells(poly, res=resolution)
        h3_indices.update(cells)

    return h3_indices

In [21]:
def build_grid_dataframe_solar(all_cells: set[str], solar_cells: set[str]) -> pd.DataFrame:
    cells = list(all_cells)
    centers = [h3.cell_to_latlng(c) for c in cells]
    return pd.DataFrame({
        "h3_index": cells,
        "lat": [c[0] for c in centers],
        "lng": [c[1] for c in centers],
        "has_solar": pd.array([c in solar_cells for c in cells], dtype="int8"),
    })

In [22]:
# Raise H3 resolution to 8 because solar farms are small
H3_RESOLUTION = 8

solar_geojson = load_dataset(SOLAR_GEOJSON_PATH)
solar_h3_cells = solar_polygons_to_h3(solar_geojson, H3_RESOLUTION)
solar_df = build_grid_dataframe_solar(generate_conus_h3_cells(H3_RESOLUTION), solar_h3_cells)
solar_df.head()

solar_df = solar_df.rename(columns={"has_turbine": "has_solar"})


In [11]:
solar_df['has_solar'].value_counts()

has_solar
0    19353142
1        2970
Name: count, dtype: int64

In [ ]:
solar_df.head()

In [18]:
import polars as pl

### Add exogenous variables
Elevation, wind speed, solar irradiance, land cover, land usage, distance to transmission lines, distance to substations, distance to roads

In [ ]:
# Polars dfs for maybe speed increases
wind_df = pl.from_pandas(wind_df)
solar_df = pl.from_pandas(solar_df)